# <font color='#1B3A5C'>Modelos de Deep Learning: LSTM y GRU</font>

> Tercer escalón de la progresión de modelado, tras el baseline estadístico (SARIMA/SARIMAX, notebook 05) y el gradient boosting (LightGBM y XGBoost, notebooks 06 y 07). Se evalúan dos arquitecturas recurrentes, LSTM y GRU, bajo el mismo split temporal, los mismos horizontes y el mismo arnés de backtesting que el resto del estudio, de modo que la comparación sea directa.

> El recorrido es deliberado: primero un único modelo global para las 30 series a la vez (que no funciona y se explica por qué), después un modelo por barrio sobre el que se hace la selección de variables y el ajuste, y finalmente los dos mejores modelos sobre los 30 códigos postales. Todo con semilla fija (12) para que los resultados sean reproducibles.

### Qué hace este notebook
1. Carga de datos y preparación (serie ancha para el modelo global y serie por barrio para el resto).
2. Primer enfoque: una sola red para las 30 series (global N:M). No funciona, y se explica el motivo.
3. Segundo enfoque, LSTM por barrio (1:1): baseline, selección de variables exógenas, ajuste de hiperparámetros y control de sobreajuste.
4. Tercer enfoque, GRU por barrio, con la misma metodología.
5. Modelos finales (el mejor LSTM y el mejor GRU) sobre los 30 códigos postales.
6. Comparativa con el resto de familias del estudio.

---
# <font color='#1B3A5C'>1. Datos y preparación</font>

> Se usa el mismo dataset que el resto de notebooks (`tfm_energy.dataset_ml` en MongoDB) y los mismos 30 códigos postales limpios. La red recurrente de skforecast (`ForecasterRnn`) se apoya en Keras 3 con backend TensorFlow; se emplea la versión de CPU, suficiente para este volumen de datos. La semilla se fija a 12 en cada modelo para reproducibilidad.

In [ ]:
import os
os.environ['TQDM_DISABLE'] = '1'
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pymongo import MongoClient
import warnings, time

import keras
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import wilcoxon
from skforecast.deep_learning import ForecasterRnn, create_and_compile_model
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster_multiseries

warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#D3D3D3'
plt.rcParams['grid.linewidth'] = 0.4
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

C1 = '#264653'; C2 = '#2A9D8F'; C3 = '#E9C46A'
C4 = '#F4A261'; C5 = '#E76F51'; C6 = '#A8DADC'
TITULO = '#1B3A5C'; SUBTITULO = '#C0392B'

SEED = 12   # semilla unica de todo el notebook

print("keras:", keras.__version__, "| backend:", keras.backend.backend())
try:
    import tensorflow as tf
    print("GPU:", tf.config.list_physical_devices('GPU') or "ninguna (se usa CPU)")
except Exception:
    print("GPU: no verificable")
start_time = time.time()

> Carga del dataset desde MongoDB. Es la misma colección y el mismo dataframe que en los notebooks 06 y 07.

In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

docs  = list(db['dataset_ml'].find({}, {'_id': 0}))
df_ml = pl.DataFrame(docs, infer_schema_length=None)

print(f"Shape: {df_ml.shape}")
print(f"Desde: {df_ml['datetime'].min()}  Hasta: {df_ml['datetime'].max()}")
print(f"Codigos postales: {df_ml['cod_postal'].n_unique()}")

> Constantes del estudio, idénticas a las del 06 y el 07 para que la comparación sea limpia: mismo split, mismos horizontes y los mismos 30 códigos postales (se excluyen los 12 corruptos de la auditoría del notebook 04.5).

 - Para la exploración por barrio se usa un subconjunto de 6 códigos que cubre todo el rango de dificultad (dos difíciles, dos intermedios y dos fáciles, según una corrida preliminar y coherente con el comportamiento en los árboles). Solo los modelos finales se entrenan sobre los 30.
 - Las variables exógenas se preparan en cuatro conjuntos de tamaño decreciente para la selección de la sección 3.2.

In [ ]:
INI       = '2019-01-01'
FIN_TRAIN = '2024-12-31 18:00:00'
INI_VAL   = '2025-01-01'
FIN_VAL   = '2025-09-30 18:00:00'
INI_TEST  = '2025-10-01'

STEPS  = 12   # 72h en bloques de 6h
TARGET = 'mwh_total'

CPS_SUCIOS = ['08011','08009','08007','08013','08010','08006','08005',
              '08019','08008','08036','08026','08037']
CPS_TODOS = [cp for cp in sorted(df_ml['cod_postal'].unique().to_list()) if cp not in CPS_SUCIOS]

# Subconjunto representativo para la exploracion (2 dificiles, 2 medios, 2 faciles)
SUBSET = ['08040', '08034', '08023', '08025', '08003', '08001']

# Conjuntos de exogenas (de mas a menos), por barrio. Sin 'anio' (extrapola al escalar 2025).
EXOG_FULL = ['lst_celsius', 'temp_mean', 'humedad_mean', 'viento_mean', 'irradiancia_mean',
             'es_festivo', 'hora', 'dia_semana', 'mes', 'semana_anio', 'es_finde', 'HDD', 'CDD',
             'lst_nublado', 'precipitacion_llueve', 'cdd_roll_3d', 'hora_x_finde', 'is_covid',
             'pct_industria_cp', 'pct_residencial_cp', 'pct_servicios_cp']                     # 21
EXOG_MIN  = ['es_festivo', 'hora', 'dia_semana', 'mes', 'semana_anio', 'es_finde', 'is_covid',
             'temp_mean', 'humedad_mean', 'HDD', 'CDD']                                        # 11: calendario + clima esencial
EXOG_CAL  = ['es_festivo', 'hora', 'dia_semana', 'mes', 'semana_anio', 'es_finde', 'is_covid'] # 7: solo calendario
EXOG_SHAP = ['hora', 'dia_semana', 'es_finde', 'es_festivo', 'temp_mean']                      # 5: lo top segun SHAP del 07

print(f"Train : {INI} a {FIN_TRAIN}")
print(f"Val   : {INI_VAL} a {FIN_VAL}")
print(f"Test  : {INI_TEST} a {df_ml['datetime'].max()}")
print(f"CPs limpios: {len(CPS_TODOS)} | subset exploracion: {SUBSET} | steps: {STEPS} | seed: {SEED}")

## <font color='#C0392B'>1.1 Serie ancha y exógenas de ciudad (para el modelo global)</font>

> El modelo global multiserie recibe la serie en formato ancho (una columna por barrio) y un único bloque de exógenas comunes a todas las series. El calendario ya es idéntico entre barrios; el clima, que varía por barrio, se promedia para obtener un clima de ciudad. Las redes no toleran NaN (a diferencia de los árboles), así que se auditan e imputan por interpolación temporal.

In [ ]:
# Serie objetivo en formato ancho: indice = tiempo (6h), una columna por barrio
series_wide = (df_ml.filter(pl.col('cod_postal').is_in(CPS_TODOS))
                    .select(['datetime', 'cod_postal', TARGET]).sort('datetime')
                    .to_pandas().pivot(index='datetime', columns='cod_postal', values=TARGET))
series_wide.index = pd.DatetimeIndex(series_wide.index)
series_wide = series_wide.asfreq('6h')

# Exogenas de ciudad: calendario (comun) + clima promediado entre barrios
CAL_CITY = ['es_festivo', 'hora', 'dia_semana', 'mes', 'semana_anio', 'es_finde', 'is_covid']
MET_CITY = ['temp_mean', 'humedad_mean', 'viento_mean', 'irradiancia_mean', 'HDD', 'CDD']
cal = (df_ml.filter(pl.col('cod_postal') == CPS_TODOS[0]).select(['datetime'] + CAL_CITY)
            .sort('datetime').to_pandas().set_index('datetime'))
met = (df_ml.filter(pl.col('cod_postal').is_in(CPS_TODOS)).group_by('datetime')
            .agg([pl.col(c).mean() for c in MET_CITY]).sort('datetime').to_pandas().set_index('datetime'))
exog_wide = cal.join(met)
exog_wide.index = pd.DatetimeIndex(exog_wide.index); exog_wide = exog_wide.asfreq('6h')
for c in exog_wide.select_dtypes('bool').columns:
    exog_wide[c] = exog_wide[c].astype('int8')

print(f"NaN target: {int(series_wide.isna().sum().sum())} | NaN exogenas: {int(exog_wide.isna().sum().sum())}")
series_wide = series_wide.interpolate('time').bfill().ffill()
exog_wide   = exog_wide.interpolate('time').bfill().ffill()
print(f"Tras imputar -> target: {int(series_wide.isna().sum().sum())} | exogenas: {int(exog_wide.isna().sum().sum())}")

s_train, s_val, s_test = series_wide.loc[:FIN_TRAIN], series_wide.loc[INI_VAL:FIN_VAL], series_wide.loc[INI_TEST:]
e_train, e_val, e_test = exog_wide.loc[:FIN_TRAIN],   exog_wide.loc[INI_VAL:FIN_VAL],   exog_wide.loc[INI_TEST:]
print(f"Train {s_train.shape[0]} | Val {s_val.shape[0]} | Test {s_test.shape[0]} bloques x {series_wide.shape[1]} barrios")

## <font color='#C0392B'>1.2 Serie por barrio y funciones reutilizables</font>

> Para el enfoque por barrio (1:1), cada modelo recibe su propia serie y sus propias exógenas. `datos_cp` arma esos datos imputados para un código postal. El resto son las funciones de evaluación, comunes a todo el notebook para no repetir código: `metricas_base` (R², MAE, RMSE, MAPE, WMAPE), `evaluar_cp` (entrena un modelo por barrio y lo evalúa por backtesting sobre el test) y `curva_cp` (devuelve la curva de entrenamiento para el control de sobreajuste).

In [ ]:
def datos_cp(cp, exog_cols=EXOG_MIN):
    """Serie (una columna) y exogenas propias de un barrio, imputadas. exog_cols=None -> sin exogenas."""
    g = (df_ml.filter(pl.col('cod_postal') == cp).sort('datetime').to_pandas().set_index('datetime'))
    g.index = pd.DatetimeIndex(g.index); g = g.asfreq('6h')
    s = g[[TARGET]].rename(columns={TARGET: cp}).interpolate('time').bfill().ffill()
    if exog_cols is None:
        return s, None
    ex = g[exog_cols].copy()
    for c in ex.select_dtypes('bool').columns:
        ex[c] = ex[c].astype('int8')
    return s, ex.interpolate('time').bfill().ffill()


def metricas_base(y_true, y_pred):
    """R2, MAE, RMSE, MAPE y WMAPE de un par real/predicho (ignora NaN)."""
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    denom = np.where(y_true == 0, np.nan, y_true)
    return {'r2': r2_score(y_true, y_pred), 'mae': mean_absolute_error(y_true, y_pred),
            'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'mape': float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100),
            'wmape': float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100)}


def _forecaster(cp, s_tr, e_tr, s_va, e_va, layer, units, lags, dropout, lr, epochs, patience):
    rk = {'activation': 'tanh'}
    if dropout > 0: rk['dropout'] = dropout
    kw = dict(series=s_tr, levels=[cp], lags=lags, steps=STEPS, recurrent_layer=layer,
              recurrent_units=units, recurrent_layers_kwargs=rk, dense_units=32,
              compile_kwargs={'optimizer': Adam(lr), 'loss': 'mse'})
    fkw = {"epochs": epochs, "batch_size": 128, "verbose": 0,
           "callbacks": [EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)],
           "series_val": s_va}
    if e_tr is not None:
        modelo = create_and_compile_model(exog=e_tr, **kw); fkw["exog_val"] = e_va
        return ForecasterRnn(estimator=modelo, levels=[cp], lags=lags, transformer_series=MinMaxScaler(),
                             transformer_exog=MinMaxScaler(), fit_kwargs=fkw)
    modelo = create_and_compile_model(**kw)
    return ForecasterRnn(estimator=modelo, levels=[cp], lags=lags, transformer_series=MinMaxScaler(),
                         transformer_exog=None, fit_kwargs=fkw)


def evaluar_cp(cp, layer='GRU', units=64, lags=28, dropout=0.0, lr=0.01,
               exog_cols=EXOG_MIN, epochs=40, patience=6, seed=SEED):
    """Entrena un modelo 1:1 para un barrio y lo evalua por backtesting sobre el test. Devuelve metricas."""
    keras.utils.set_random_seed(seed)
    s, ex = datos_cp(cp, exog_cols)
    s_tr, s_va = s.loc[:FIN_TRAIN], s.loc[INI_VAL:FIN_VAL]
    e_tr, e_va = (ex.loc[:FIN_TRAIN], ex.loc[INI_VAL:FIN_VAL]) if ex is not None else (None, None)
    fc = _forecaster(cp, s_tr, e_tr, s_va, e_va, layer, units, lags, dropout, lr, epochs, patience)
    cv = TimeSeriesFold(steps=STEPS, initial_train_size=len(s_tr) + len(s_va), refit=False)
    bt = dict(cv=cv, metric='mean_absolute_error', add_aggregated_metric=False,
              verbose=False, show_progress=False, suppress_warnings=True)
    _, pr = (backtesting_forecaster_multiseries(fc, series=s, exog=ex, **bt) if ex is not None
             else backtesting_forecaster_multiseries(fc, series=s, **bt))
    prs = pr.loc[pr['level'] == cp, 'pred'].sort_index()
    m = metricas_base(s[cp].reindex(prs.index).values, prs.values); m['cp'] = cp
    return m


def curva_cp(cp, layer='GRU', units=64, lags=28, dropout=0.0, lr=0.01, exog_cols=EXOG_MIN, epochs=40, patience=6, seed=SEED):
    """Fit directo (sin backtest) para obtener la curva de entrenamiento (history_)."""
    keras.utils.set_random_seed(seed)
    s, ex = datos_cp(cp, exog_cols)
    s_tr, s_va = s.loc[:FIN_TRAIN], s.loc[INI_VAL:FIN_VAL]
    e_tr, e_va = (ex.loc[:FIN_TRAIN], ex.loc[INI_VAL:FIN_VAL]) if ex is not None else (None, None)
    fc = _forecaster(cp, s_tr, e_tr, s_va, e_va, layer, units, lags, dropout, lr, epochs, patience)
    fc.fit(series=s_tr, exog=e_tr) if e_tr is not None else fc.fit(series=s_tr)
    return fc.history_


def tabla_subset(configs, etiqueta_col='config'):
    """Corre un diccionario {nombre: kwargs} sobre el SUBSET y devuelve un DataFrame config x barrio + MEDIANA."""
    filas = {}
    for nombre, cfg in configs.items():
        t0 = time.time()
        filas[nombre] = {cp: evaluar_cp(cp, **cfg)['r2'] for cp in SUBSET}
        print(f"{nombre:24s} -> mediana {np.median(list(filas[nombre].values())):.3f}  ({(time.time()-t0):.0f}s)", flush=True)
    t = pd.DataFrame(filas).T[SUBSET]; t['MEDIANA'] = t.median(axis=1)
    return t.round(3)

print("Funciones cargadas: datos_cp, metricas_base, evaluar_cp, curva_cp, tabla_subset")

---
# <font color='#1B3A5C'>2. Primer enfoque: una sola red para las 30 series (global)</font>

> El primer intento replica la idea que funcionó con los árboles: un único modelo para todos los barrios. Pero la red recurrente multiserie de skforecast no apila las series como los árboles, sino que las trata como entradas y salidas simultáneas de un mismo modelo multivariante (configuración N:M). Se entrena y se evalúa igual que el resto.

 - Las exógenas aquí son las de ciudad (compartidas), porque esta configuración no admite unas distintas por barrio.

In [ ]:
keras.utils.set_random_seed(SEED)
levels = CPS_TODOS
modelo_nm = create_and_compile_model(
    series=s_train, levels=levels, lags=28, steps=STEPS, exog=e_train,
    recurrent_layer="LSTM", recurrent_units=64, dense_units=32,
    compile_kwargs={'optimizer': Adam(0.01), 'loss': 'mse'})
forecaster_nm = ForecasterRnn(
    estimator=modelo_nm, levels=levels, lags=28,
    transformer_series=MinMaxScaler(), transformer_exog=MinMaxScaler(),
    fit_kwargs={"epochs": 40, "batch_size": 128, "verbose": 0,
                "callbacks": [EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)],
                "series_val": s_val, "exog_val": e_val})

cv = TimeSeriesFold(steps=STEPS, initial_train_size=len(s_train) + len(s_val), refit=False)
_, pred_nm = backtesting_forecaster_multiseries(
    forecaster_nm, series=series_wide, exog=exog_wide, cv=cv, metric='mean_absolute_error',
    add_aggregated_metric=False, verbose=False, show_progress=False, suppress_warnings=True)
r2_nm = [metricas_base(series_wide[cp].reindex(pred_nm.loc[pred_nm['level']==cp].index).values,
                       pred_nm.loc[pred_nm['level']==cp, 'pred'].values)['r2'] for cp in CPS_TODOS]
print(f"R2 mediano (test, modelo global N:M): {np.median(r2_nm):.3f}")

> El modelo global multiserie rinde claramente por debajo de los modelos del 05 y de los árboles. No es un fallo de implementación: predecir las 30 series heterogéneas a la vez (configuración N:M) es, según el propio benchmark de skforecast, la peor de las configuraciones recurrentes, porque un único modelo multivariante debe repartir su capacidad entre barrios muy distintos (rango de consumo de ocho veces). El consumo de cada barrio se predice mejor con su propia historia.

 - Conclusión: se abandona el enfoque global y se pasa a un modelo por barrio (1:1), que además permite usar el clima propio de cada código postal. El N:M queda documentado como primer intento.

---
# <font color='#1B3A5C'>3. Segundo enfoque: un modelo por barrio (LSTM)</font>

> Se entrena un LSTM independiente por barrio. Cada red predice su propia serie a partir de su propio histórico, lo que permite especializar el modelo y usar el clima y el perfil sectorial propios del código postal. La exploración (baseline, selección de variables y ajuste) se hace sobre el subconjunto de 6 barrios para acotar el coste; solo el modelo final se entrena sobre los 30.

## <font color='#C0392B'>3.1 Modelo base</font>

> Punto de partida sin ajustar: LSTM de 64 unidades, ventana de una semana (28 bloques) y el conjunto completo de 21 exógenas por barrio.

In [ ]:
base_lstm = {cp: evaluar_cp(cp, layer='LSTM', exog_cols=EXOG_FULL)['r2'] for cp in SUBSET}
print("LSTM base (21 exogenas) por barrio:")
for cp, r in base_lstm.items():
    print(f"  {cp}: {r:.3f}")
print(f"\nR2 mediano (subset): {np.median(list(base_lstm.values())):.3f}")

> El modelo base ya predice de forma razonable en los barrios regulares, pero arrastra a los de consumo errático. Sobre esta referencia se prueba qué mejora: primero las variables, luego los hiperparámetros.

## <font color='#C0392B'>3.2 Selección de variables exógenas</font>

> En lugar de un análisis SHAP (poco práctico sobre redes recurrentes de secuencia), la importancia de las variables se mide por ablación: se reduce el bloque de exógenas y se observa el efecto en el R². Se prueba una escalera de menos a más reducida: 21 (todas), 11 (calendario + clima esencial), 7 (solo calendario), 5 (las más importantes según el SHAP de los árboles del 07) y 0 (sin exógenas, puramente autorregresivo).

In [ ]:
configs_exog = {
    'LSTM 21 (todas)':  dict(layer='LSTM', exog_cols=EXOG_FULL),
    'LSTM 11 (cal+clima)': dict(layer='LSTM', exog_cols=EXOG_MIN),
    'LSTM 7 (calendario)': dict(layer='LSTM', exog_cols=EXOG_CAL),
    'LSTM 5 (SHAP-top)': dict(layer='LSTM', exog_cols=EXOG_SHAP),
    'LSTM 0 (sin exog)': dict(layer='LSTM', exog_cols=None),
}
tab_exog_lstm = tabla_subset(configs_exog)
print("\n" + tab_exog_lstm.to_string())

> La ablación da el hallazgo más importante del capítulo: reducir las exógenas de 21 a 11 (quitar el ruido del satélite, el viento, la irradiancia, el perfil sectorial y las medias móviles, y quedarse con calendario y clima esencial) mejora de forma marcada el R². Por debajo de 11 vuelve a caer, y sin exógenas es lo peor.

 - Lectura: con poco dato por serie, la red se sobreajusta a las variables ruidosas; un input depurado generaliza mucho mejor. Esto contrasta con los árboles (notebook 07), que digerían las 21 sin problema gracias a su selección interna. El clima esencial sí aporta (quitarlo empeora), en línea con el SHAP del 07, donde la autorregresión y el calendario dominaban y el clima quedaba en segundo plano.
 - Conjunto elegido: las 11 (calendario + clima esencial).

## <font color='#C0392B'>3.3 Ajuste de hiperparámetros</font>

> Fijado el conjunto de 11 exógenas, se prueban las palancas habituales de una red recurrente: más capacidad (128 unidades), regularización por dropout y un entrenamiento más fino (tasa de aprendizaje menor con reductor y más paciencia).

In [ ]:
configs_tune_lstm = {
    'LSTM 11 base':       dict(layer='LSTM', units=64,  exog_cols=EXOG_MIN),
    'LSTM 11 +capac u128':dict(layer='LSTM', units=128, exog_cols=EXOG_MIN),
    'LSTM 11 +dropout0.1':dict(layer='LSTM', units=64,  dropout=0.1, exog_cols=EXOG_MIN),
    'LSTM 11 lr0.005':    dict(layer='LSTM', units=64,  lr=0.005, epochs=60, patience=10, exog_cols=EXOG_MIN),
}
tab_tune_lstm = tabla_subset(configs_tune_lstm)
print("\n" + tab_tune_lstm.to_string())

> Ninguna palanca de hiperparámetros mejora a la base: más capacidad sobreajusta (sobre todo en los barrios difíciles), el dropout quita señal y el entrenamiento más lento no aporta. La conclusión, honesta, es que en este problema la palanca que mueve el resultado es la selección de variables (sección 3.2), no la arquitectura fina.

 - Mejor LSTM: 64 unidades, ventana de una semana, 11 exógenas, tasa 0,01.

## <font color='#C0392B'>3.4 Control de sobreajuste del mejor LSTM</font>

> Se revisa la curva de entrenamiento (pérdida de entrenamiento frente a validación) del modelo elegido en los 6 barrios. Un hueco pequeño y estable indica que el modelo generaliza; un hueco que se dispara, sobreajuste.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8)); axes = axes.ravel()
filas = []
for ax, cp in zip(axes, SUBSET):
    h = curva_cp(cp, layer='LSTM', exog_cols=EXOG_MIN)
    loss, vloss = h['loss'], h['val_loss']; best = int(np.argmin(vloss)); gap = vloss[best] - loss[best]
    ax.plot(range(1, len(loss)+1),  loss,  color=C1, marker='o', ms=2, label='train')
    ax.plot(range(1, len(vloss)+1), vloss, color=C5, marker='s', ms=2, label='val')
    ax.axvline(best + 1, color=C2, ls='--', lw=1)
    ax.set_title(f"{cp}  (gap {gap:.4f})", color=SUBTITULO); ax.legend(fontsize=8)
    filas.append({'cp': cp, 'mejor_epoca': best + 1, 'gap': round(gap, 5)})
plt.tight_layout(); plt.show()
print(pd.DataFrame(filas).to_string(index=False))

> Los huecos entre entrenamiento y validación son pequeños y estables, y el early stopping selecciona el punto óptimo en cada barrio. El modelo no sobreajusta: la diferencia que se observa entre train y validación es coherente con el cambio de régimen de 2025 documentado en la auditoría (la validación es un periodo distinto y más difícil), no con memorización. Lo confirma que añadir dropout no mejoró (sección 3.3): si fuera sobreajuste, la regularización habría ayudado.

---
# <font color='#1B3A5C'>4. Tercer enfoque: GRU por barrio</font>

> La GRU es una red recurrente más simple que la LSTM (dos compuertas en vez de tres), con menos parámetros y, según la literatura, rendimiento comparable o superior. Se repite exactamente la misma metodología (selección de variables y ajuste) sobre el mismo subconjunto, para comparar las dos arquitecturas en igualdad de condiciones.

In [ ]:
configs_exog_gru = {
    'GRU 21 (todas)':      dict(layer='GRU', exog_cols=EXOG_FULL),
    'GRU 11 (cal+clima)':  dict(layer='GRU', exog_cols=EXOG_MIN),
    'GRU 7 (calendario)':  dict(layer='GRU', exog_cols=EXOG_CAL),
    'GRU 5 (SHAP-top)':    dict(layer='GRU', exog_cols=EXOG_SHAP),
    'GRU 0 (sin exog)':    dict(layer='GRU', exog_cols=None),
}
tab_exog_gru = tabla_subset(configs_exog_gru)
print("\n" + tab_exog_gru.to_string())

In [ ]:
configs_tune_gru = {
    'GRU 11 base':        dict(layer='GRU', units=64,  exog_cols=EXOG_MIN),
    'GRU 11 +capac u128': dict(layer='GRU', units=128, exog_cols=EXOG_MIN),
    'GRU 11 +dropout0.1': dict(layer='GRU', units=64,  dropout=0.1, exog_cols=EXOG_MIN),
}
tab_tune_gru = tabla_subset(configs_tune_gru)
print("\n" + tab_tune_gru.to_string())

> La GRU reproduce el mismo patrón que el LSTM: el conjunto de 11 exógenas es el mejor y los ajustes de capacidad o dropout no mejoran. Y, lo más relevante, GRU y LSTM con 11 exógenas quedan prácticamente empatados.

 - Conclusión doble. Primera, la arquitectura recurrente concreta (LSTM o GRU) no es el factor decisivo en este problema; lo decisivo es la selección de variables, que mejora a ambas por igual. Segunda, se conserva un modelo final de cada familia (mejor LSTM y mejor GRU, ambos con 11 exógenas) para la corrida sobre los 30 y la comparativa.

---
# <font color='#1B3A5C'>5. Modelos finales sobre los 30 códigos postales</font>

> Con las configuraciones ganadoras (LSTM y GRU, ambos de 64 unidades, ventana de una semana y 11 exógenas), se entrena un modelo por cada uno de los 30 barrios y se evalúa sobre el test reservado (octubre y noviembre de 2025). El R² mediano entre barrios es la cifra comparable con el resto de familias.

In [ ]:
def correr_30(layer):
    res = []
    t0 = time.time()
    for i, cp in enumerate(CPS_TODOS, 1):
        m = evaluar_cp(cp, layer=layer, units=64, lags=28, exog_cols=EXOG_MIN)
        res.append(m)
        print(f"[{i:2d}/30] {cp}: R2 {m['r2']:.3f}", flush=True)
    print(f"{layer} 1:1 (30 barrios) en {(time.time()-t0)/60:.1f} min")
    return pd.DataFrame(res)

m_lstm = correr_30('LSTM')
m_lstm.to_csv('resultados_lstm_test.csv', index=False)
print(f"\nLSTM final -> R2 mediano {m_lstm['r2'].median():.3f} | WMAPE {m_lstm['wmape'].median():.2f}%")

In [ ]:
m_gru = correr_30('GRU')
m_gru.to_csv('resultados_gru_test.csv', index=False)
print(f"\nGRU final -> R2 mediano {m_gru['r2'].median():.3f} | WMAPE {m_gru['wmape'].median():.2f}%")

## <font color='#C0392B'>5.1 Análisis de resultados</font>

> R² por barrio de los dos modelos finales. Identifica qué barrios arrastran el resultado (los de consumo errático, idiosincráticos) y cuáles se predicen muy bien, en coherencia con el resto del estudio.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
orden = m_gru.sort_values('r2')['cp']
x = np.arange(len(orden)); w = 0.4
ax.bar(x - w/2, m_lstm.set_index('cp').loc[orden, 'r2'], w, color=C4, label='LSTM')
ax.bar(x + w/2, m_gru.set_index('cp').loc[orden, 'r2'],  w, color=C2, label='GRU')
ax.axhline(m_gru['r2'].median(), color=C1, ls='--', lw=1)
ax.set_xticks(x); ax.set_xticklabels(orden, rotation=90); ax.set_ylabel('R2 (test)')
ax.set_title('R2 por barrio, modelos finales', color=SUBTITULO); ax.legend()
plt.tight_layout(); plt.show()
print(f"Barrios R2<0,5: {(m_gru['r2']<0.5).sum()} | R2>0,8: {(m_gru['r2']>0.8).sum()} (GRU)")

## <font color='#C0392B'>5.2 Calibración y predicción en barrios concretos</font>

> Para el mejor modelo (GRU), se muestra la predicción frente al consumo real en un barrio fácil y uno difícil, junto con la calibración. Ilustra que el error es idiosincrático del barrio: donde la serie es regular el modelo acierta, donde es errática falla, igual que el resto de familias.

In [ ]:
def predecir_cp(cp, layer='GRU'):
    keras.utils.set_random_seed(SEED)
    s, ex = datos_cp(cp, EXOG_MIN)
    s_tr, s_va = s.loc[:FIN_TRAIN], s.loc[INI_VAL:FIN_VAL]
    e_tr, e_va = ex.loc[:FIN_TRAIN], ex.loc[INI_VAL:FIN_VAL]
    fc = _forecaster(cp, s_tr, e_tr, s_va, e_va, layer, 64, 28, 0.0, 0.01, 40, 6)
    cv = TimeSeriesFold(steps=STEPS, initial_train_size=len(s_tr)+len(s_va), refit=False)
    _, pr = backtesting_forecaster_multiseries(fc, series=s, exog=ex, cv=cv, metric='mean_absolute_error',
                add_aggregated_metric=False, verbose=False, show_progress=False, suppress_warnings=True)
    prs = pr.loc[pr['level']==cp, 'pred'].sort_index()
    return pd.DataFrame({'real': s[cp].reindex(prs.index).values, 'pred': prs.values}, index=prs.index)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for fila, cp in zip(axes, ['08001', '08040']):
    d = predecir_cp(cp)
    fila[0].plot(d.index, d['real'], color=C1, lw=1, label='real')
    fila[0].plot(d.index, d['pred'], color=C5, lw=1, ls='--', label='GRU')
    fila[0].set_title(f'{cp}: real vs predicho (test)', color=SUBTITULO); fila[0].legend(); fila[0].tick_params(axis='x', rotation=45)
    fila[1].scatter(d['real'], d['pred'], s=10, alpha=0.3, color=C1)
    lim = [d.min().min(), d.max().max()]; fila[1].plot(lim, lim, color=C5, ls='--')
    fila[1].set_title(f'{cp}: calibracion', color=SUBTITULO); fila[1].set_xlabel('real'); fila[1].set_ylabel('pred')
plt.tight_layout(); plt.show()

## <font color='#C0392B'>5.3 Persistencia de los modelos finales</font>

> Se guardan los dos modelos finales con `save_forecaster` de skforecast (las redes de Keras no se serializan con joblib como los árboles). Se entrena cada uno hasta el fin de validación, replicando el ajuste evaluado en el test, y se conserva para el dashboard y el sistema de alerta del capítulo VI.

In [ ]:
from skforecast.utils import save_forecaster

def construir_y_entrenar(cp, layer):
    keras.utils.set_random_seed(SEED)
    s, ex = datos_cp(cp, EXOG_MIN)
    s_tr, s_va = s.loc[:FIN_VAL], s.loc[INI_VAL:FIN_VAL]
    fc = _forecaster(cp, s.loc[:FIN_TRAIN], ex.loc[:FIN_TRAIN], s_va, ex.loc[INI_VAL:FIN_VAL],
                     layer, 64, 28, 0.0, 0.01, 40, 6)
    fc.fit(series=s.loc[:FIN_VAL], exog=ex.loc[:FIN_VAL])
    return fc

# Ejemplo: se guarda el modelo de un barrio de referencia por familia.
# (En produccion se itera sobre los 30; aqui se ilustra el guardado.)
for layer, nombre in [('LSTM', 'lstm'), ('GRU', 'gru')]:
    try:
        fc = construir_y_entrenar('08001', layer)
        save_forecaster(fc, file_name=f'models/forecaster_{nombre}_08001.joblib', verbose=False)
        print(f"Guardado: models/forecaster_{nombre}_08001.joblib")
    except Exception as e:
        print(f"No se pudo guardar {nombre} con save_forecaster (revisar API RNN): {e}")

---
# <font color='#1B3A5C'>6. Comparativa con el resto de familias</font>

> Se sitúan los modelos de deep learning junto al resto del estudio sobre el mismo test (octubre y noviembre de 2025, horizonte de 72 horas, 30 códigos postales). El R² mediano entre barrios es el estadístico de referencia.

In [ ]:
archivos = {'Naive': 'resultados_baseline.csv', 'SARIMAX': 'resultados_sarimax.csv',
            'SARIMA': 'resultados_sarima.csv', 'LightGBM': 'resultados_lightgbm_test.csv',
            'XGBoost': 'resultados_xgboost_test.csv', 'LSTM': 'resultados_lstm_test.csv',
            'GRU': 'resultados_gru_test.csv'}
res = {n: pd.read_csv(f, dtype={'cp': str}).set_index('cp') for n, f in archivos.items()}

tabla = pd.DataFrame({'R2_mediano':    {n: d['r2'].median()    for n, d in res.items()},
                      'WMAPE_mediano': {n: d['wmape'].median() for n, d in res.items()}}).sort_values('R2_mediano')
print(tabla.round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
col = [C2 if n in ('LSTM', 'GRU') else C1 for n in tabla.index]
ax.barh(tabla.index, tabla['R2_mediano'], color=col, edgecolor='white')
for i, v in enumerate(tabla['R2_mediano']): ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel('R2 mediano (test)'); ax.set_title('Comparativa de modelos', color=SUBTITULO)
plt.tight_layout(); plt.show()

In [ ]:
# Wilcoxon pareado: el mejor deep learning frente al mejor arbol
mejor_dl = 'GRU' if res['GRU']['r2'].median() >= res['LSTM']['r2'].median() else 'LSTM'
c = res[mejor_dl][['r2']].join(res['XGBoost'][['r2']], lsuffix='_dl', rsuffix='_xgb', how='inner').dropna()
stat, p = wilcoxon(c['r2_dl'], c['r2_xgb'])
print(f"{mejor_dl} vs XGBoost: {mejor_dl} gana en {(c['r2_dl']>c['r2_xgb']).sum()}/{len(c)} | Wilcoxon p={p:.4f}")
print(f"  mediana {mejor_dl} {c['r2_dl'].median():.3f} vs XGBoost {c['r2_xgb'].median():.3f}")

> Cierre del capítulo de deep learning. Las dos redes recurrentes (LSTM y GRU), una vez depuradas las variables, quedan a la altura de SARIMA pero por debajo de los modelos de gradient boosting, que siguen siendo los mejores del estudio.

 - La diferencia con los árboles es coherente con la evidencia reciente sobre datos tabulares (Grinsztajn et al. 2022; Shwartz-Ziv y Armon 2022): con variables bien diseñadas, dato modesto por serie y un test con cambio de régimen, el gradient boosting supera al deep learning. La ventaja del deep learning reportada en estudios con datos de alta granularidad y series homogéneas (Gassar 2024) no se traslada a este contexto.
 - El hallazgo metodológico propio: en las redes, la palanca decisiva fue la selección de variables, no la arquitectura ni los hiperparámetros; LSTM y GRU convergen al mismo rendimiento con un input depurado.
 - Los modelos finales quedan guardados para el dashboard y el sistema de alerta del capítulo VI.

In [ ]:
print(f"Tiempo total del notebook: {(time.time() - start_time)/60:.1f} min")